# Eye-Tracking Decision Strategy Analysis - Validation Notebook

**Purpose:**
This notebook validates the exported fixation tables and reproduces key movement metrics in Python.

**What this notebook does / does not do:**
- **DOES:** Validate fixation metrics, compute scan index (H/V),and visualize distributions.
- **DOES NOT:** Run the raw EyeLink parsing or pupil filtering.

**Expected inputs:** CSV outputs from the MATLAB pipeline (e.g., `Subject_XXX_detailed.csv`).
**Where outputs will be saved:** `outputs/validation/`.
**Version:** 1.0

---

## 1) Project Context and Pipeline Overview
This project analyzes decision strategies under time pressure using a multi-attribute choice task. Blocks vary by the number of attributes.
Data flows from `EyeLink` -> `.mat` (edfStruct) -> `Convert_eye_data.m` -> CSV outputs -> `analyze_movements.py` / `classifyScanHV_updated.m`.

**Pipeline Components:**
- `DM_main_noa.m`: The main experiment script.
- `Convert_eye_data.m`: Exports tables.
- `analyze_movements.py`: Computes movement metrics and transitions.
- `classifyScanHV_updated.m`: HV scan index from AOI sequences in MATLAB.

## 2) Definitions and Vocabulary (Critical)
- **Fixation:** A stable gaze sample cluster; exported as a row in `*_detailed.csv`.
- **Saccade / Movement time:** The time between a fixation's end and the next fixation's start.
- **AOI:** Discrete region label (e.g., `A1`, `B3`).
- **Transition:** Consecutive pair (`AOI_t` -> `AOI_{t+1}`).
- **Horizontal scan (H):** Same row/attribute across alternatives.
- **Vertical scan (V):** Same alternative across attributes.
- **Scan Index:** Explicitly `H / (H + V)`.
- **Other transitions:** Same cell repeated, header-to-cell, NaN, or outside valid AOIs.

## 3) Data Files and Schemas
**File:** `Subject_XXX_detailed.csv` (long fixation-level table)

**Expected columns:**
- *Required:* Block, Trial, FixationIndex, AOI, StartTime, EndTime, Duration, X, Y
- *Optional:* SD_X, SD_Y, Pupil, Validity
> **Rule:** If a required column is missing, the notebook will warn and skip dependent analyses.

## 4) Reproducibility Setup
- **Environment:** Launch from project root via `uv run jupyter notebook`.
- **Execution:** Run all cells from top to bottom.

In [1]:
import os
import sys
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

# 5) Load + Subject/Batch Selection
print('This notebook can run on one subject or a batch.')

PROJECT_ROOT = Path().absolute()
DATA_DIR = PROJECT_ROOT / 'data' / 'processed' / 'fixations'
OUTPUTS_DIR = PROJECT_ROOT / 'outputs' / 'validation'
OUTPUTS_DIR.mkdir(parents=True, exist_ok=True)

SUBJECT_ID = '661'   # <-- Change this to specific subject code. for example '889'
FILE_PATH = DATA_DIR / f'{SUBJECT_ID}_detailed.csv'

print("Looking for file:", FILE_PATH)


if not FILE_PATH.exists():
    print(f'WARNING: File {FILE_PATH} not found. Using MOCK data for demonstration.')
    df = pd.DataFrame({
        'Block': [1, 1, 1, 1, 1, 2, 2, 2],
        'Trial': [1, 1, 1, 1, 2, 1, 1, 1],
        'FixationIndex': [1, 2, 3, 4, 1, 1, 2, 3],
        'AOI': ['A1', 'B1', 'B2', np.nan, 'A1', 'C3', 'C3', 'A3'],
        'StartTime': [100, 350, 700, 1000, 1500, 2000, 2300, 2700],
        'EndTime': [300, 600, 950, 1200, 1800, 2200, 2500, 3000],
        'Duration': [200, 250, 250, 200, 300, 200, 200, 300]
    })
else:
    df = pd.read_csv(FILE_PATH)

display(df.head())
display(df.info())

This notebook can run on one subject or a batch.
Looking for file: c:\Projects\Thesis_EyeTracking\data\processed\fixations\661_detailed.csv


,Block,Trial,FixationIndex,AOI,StartTime,EndTime,Duration
0,1,1,1,A1,100,300,200
1,1,1,2,B1,350,600,250
2,1,1,3,B2,700,950,250
3,1,1,4,NaN,1000,1200,200
4,1,2,1,A1,1500,1800,300


<class 'pandas.DataFrame'>
RangeIndex: 8 entries, 0 to 7
Data columns (total 7 columns):
 #   Column         Non-Null Count  Dtype
---  ------         --------------  -----
 0   Block          8 non-null      int64
 1   Trial          8 non-null      int64
 2   FixationIndex  8 non-null      int64
 3   AOI            7 non-null      str  
 4   StartTime      8 non-null      int64
 5   EndTime        8 non-null      int64
 6   Duration       8 non-null      int64
dtypes: int64(6), str(1)
memory usage: 580.0 bytes


None

## 6) Core Data Integrity Checks (Sanity Checks)
We verify data integrity before calculating any metrics to ensure validity.

In [ ]:
print('--- Data Integrity Checks ---')

nan_aoi_pct = df['AOI'].isna().mean()
print(f'Missing AOIs: {df["AOI"].isna().sum()} ({nan_aoi_pct:.1%})')

df['TimeDiff'] = df['StartTime_ms'] - df['StartTime_ms'].shift(1)
non_monotonic = (df['TimeDiff'] < 0) & (df['Trial'] == df['Trial'].shift(1))
print(f'Non-monotonic times in same trial: {non_monotonic.sum()}')

print(f'Negative durations: {(df["Duration_ms"] < 0).sum()}')
print(f'Duplicated rows: {df.duplicated().sum()}')

gap = (df['StartTime_ms'] - df['EndTime_ms'].shift(1))
valid_gaps = gap[df['Trial'] == df['Trial'].shift(1)].dropna()

summary_table = pd.DataFrame([{
    'N Fixations': len(df),
    'N Trials': df['Trial'].nunique(),
    'N Blocks': df['Block'].nunique(),
    '% NaN AOI': f'{nan_aoi_pct:.1%}',
    'Min Duration': df['Duration_ms'].min(),
    'Median Duration': df['Duration_ms'].median(),
    'Max Duration': df['Duration_ms'].max(),
    'Min Inter-fixation Gap': valid_gaps.min() if not valid_gaps.empty else np.nan,
    'Median Inter-fixation Gap': valid_gaps.median() if not valid_gaps.empty else np.nan,
}])

display(summary_table)

--- Data Integrity Checks ---
Missing AOIs: 4 (9.8%)
Non-monotonic times in same trial: 0
Negative durations: 0
Duplicated rows: 0


,N Fixations,N Trials,N Blocks,% NaN AOI,Min Duration,Median Duration,Max Duration,Min Inter-fixation Gap,Median Inter-fixation Gap
0,41,2,2,9.8%,50,280.0,923,0.0,23.0


## 7) Derived Features (Computed Columns)
Computing `NextAOI`, `NextStartTime`, and `GapToNext` (movement proxy).

## 8) Explicit Scanpath Classification Rules
- **Horizontal:** Same attribute index, different alternative letter (`A1` -> `B1`).
- **Vertical:** Same alternative letter, different attribute index (`A1` -> `A2`).
- **Ignored:** NaN AOIs, repeated AOIs (`A1` -> `A1`), header transitions.

In [13]:
def classify_transition(aoi1, aoi2):
    if pd.isna(aoi1) or pd.isna(aoi2):
        return 'Ignored'
    aoi1, aoi2 = str(aoi1).strip(), str(aoi2).strip()
    if aoi1 == aoi2:
        return 'Ignored'
    
    if len(aoi1) >= 2 and len(aoi2) >= 2:
        alt1, attr1 = aoi1[0], aoi1[1:]
        alt2, attr2 = aoi2[0], aoi2[1:]
        
        if alt1.isalpha() and alt2.isalpha() and attr1.isdigit() and attr2.isdigit():
            if attr1 == attr2 and alt1 != alt2:
                return 'Horizontal'
            elif alt1 == alt2 and attr1 != attr2:
                return 'Vertical'
    return 'Other'

df['NextAOI'] = df.groupby(['Block', 'Trial'])['AOI'].shift(-1)
df['NextStartTime'] = df.groupby(['Block', 'Trial'])['StartTime'].shift(-1)
df['GapToNext'] = df['NextStartTime'] - df['EndTime']
df['TransitionType'] = df.apply(lambda row: classify_transition(row['AOI'], row['NextAOI']), axis=1)

display(df[['Block', 'Trial', 'AOI', 'NextAOI', 'TransitionType', 'GapToNext']].head(10))

KeyError: 'Column not found: StartTime'

## 9) Trial-Level and Block-Level Metrics
Aggregating counts and calculating the **Scan Index** per trial and block.

In [ ]:
trial_metrics = df.groupby(['Block', 'Trial']).agg(
    H_count=('TransitionType', lambda x: (x == 'Horizontal').sum()),
    V_count=('TransitionType', lambda x: (x == 'Vertical').sum()),
    Other_count=('TransitionType', lambda x: (x == 'Other').sum()),
    Ignored_count=('TransitionType', lambda x: (x == 'Ignored').sum()),
    Total_Fixation_Time=('Duration', 'sum'),
    Mean_Fix_Duration=('Duration', 'mean'),
    Mean_GapToNext=('GapToNext', 'mean')
).reset_index()

# Scan Index = H / (H + V). If H+V=0, it becomes NaN.
trial_metrics['ScanIndex'] = np.where(
    (trial_metrics['H_count'] + trial_metrics['V_count']) > 0,
    trial_metrics['H_count'] / (trial_metrics['H_count'] + trial_metrics['V_count']),
    np.nan
)

block_metrics = trial_metrics.groupby('Block').agg(
    Mean_ScanIndex=('ScanIndex', 'mean'),
    Mean_H=('H_count', 'mean'),
    Mean_V=('V_count', 'mean')
).reset_index()

display(trial_metrics.head())
display(block_metrics)

## 10) Visualization Section
Plots to verify distributions and strategy shifts.

In [1]:
sns.set_theme(style='whitegrid')
fig, axes = plt.subplots(2, 3, figsize=(18, 10))
fig.suptitle(f'Validation Plots for {SUBJECT_ID}', fontsize=16)

# 1. Bar chart: counts
sns.countplot(data=df, x='TransitionType', ax=axes[0,0], palette='viridis')
axes[0,0].set_title('Transition Counts')

# 2. Histogram: Durations
sns.histplot(df['Duration'].dropna(), bins=30, kde=True, ax=axes[0,1], color='coral')
axes[0,1].set_title('Fixation Durations')

# 3. Histogram: Gaps
sns.histplot(df['GapToNext'].dropna(), bins=30, kde=True, ax=axes[0,2], color='teal')
axes[0,2].set_title('Movement Gaps (GapToNext)')

# 4. Line plot across trials
sns.lineplot(data=trial_metrics, x='Trial', y='ScanIndex', hue='Block', marker='o', ax=axes[1,0])
axes[1,0].set_title('Trial Scan Index')
axes[1,0].set_ylim(-0.1, 1.1)

# 5. Boxplot by block
sns.boxplot(data=trial_metrics, x='Block', y='ScanIndex', ax=axes[1,1], palette='Set2')
axes[1,1].set_title('Scan Index by Block')

# 6. Heatmap / Barplot: AOI frequency
aoi_counts = df['AOI'].value_counts().reset_index()
if not aoi_counts.empty and 'AOI' in aoi_counts.columns:
    sns.barplot(data=aoi_counts.head(10), x='AOI', y='count', ax=axes[1,2], palette='mako')
    axes[1,2].set_title('Top 10 Visited AOIs')
    axes[1,2].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()

NameError: name 'sns' is not defined

## 11) Validation Against MATLAB Outputs (Crucial)
Comparing Python metrics against the `.mat` derived summary.

In [ ]:
# Mocking MATLAB summary output for demonstration
matlab_df = pd.DataFrame({
    'Block': [1, 2],
    'Matlab_ScanIndex': [1.0, 0.0]  # Replace with actual imported values
})

validation = pd.merge(block_metrics[['Block', 'Mean_ScanIndex']], matlab_df, on='Block', how='left')
validation['Delta'] = (validation['Mean_ScanIndex'] - validation['Matlab_ScanIndex']).abs()

display(validation)

if (validation['Delta'] > 1e-6).any():
    print('WARNING: Mismatch detected. Please check filtering rules.')
else:
    print('SUCCESS: Python metrics match MATLAB outputs!')

## 12) Edge Cases & How They Are Handled
- **Sequences with NaN gaps:** Transitions involving NaN AOIs are classified as 'Ignored'.
- **H+V=0:** Scan index is set to `NaN` (not 0) and excluded from block means.
- **Repeated AOIs:** Marked as 'Ignored'.

## 13) Interpretation Guide
- **High scan index (~1):** More horizontal (attribute-wise comparison).
- **Low scan index (~0):** More vertical (alternative-wise evaluation).

## 14) How to Extend (Future Steps)
- Weighting transitions by fixation duration.
- Compute dwell time per AOI.
- Strategy clustering across subjects.

## 15) Notebook Outputs
Saving the validated metrics and generated figures.

In [ ]:
summary_csv = OUTPUTS_DIR / f'{SUBJECT_ID}_validation_summary.csv'
trial_metrics.to_csv(summary_csv, index=False)
print(f'Saved metrics to: {summary_csv}')

fig_path = OUTPUTS_DIR / f'{SUBJECT_ID}_validation_plots.png'
fig.savefig(fig_path, dpi=300)
print(f'Saved figures to: {fig_path}')

## 16) Final Summary
**Key Results:**
- Data Quality: [OK / Review Required]
- Overall strategy tendency: [Horizontal / Vertical / Mixed]
- Next steps: Ready for multi-subject batch aggregation.